# Train Aira on Kaggle (xl / xxl / titan presets)

Kaggle setup:
1. Upload this repo (or add it as a Kaggle Dataset) and attach it to the notebook.
2. Settings → Accelerator → **GPU T4 x2**.
3. Run the cells below, top to bottom.
- Both T4s are used automatically (DataParallel) - no code change needed.


In [ ]:
import subprocess, sys, torch
n = torch.cuda.device_count()
print('GPUs:', n, [torch.cuda.get_device_name(i) for i in range(n)] or 'NONE - enable GPU in Settings!')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy'], check=False)

In [ ]:
import os
REPO = '/kaggle/working/aira-agi'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone',
                    'https://github.com/iam169459/aira-agi.git', REPO], check=True)
os.chdir(REPO)
print('repo ready:', os.getcwd())

In [ ]:
# One model that grows: run again later and it resumes from where it stopped.
# Resume compatibility checks architecture + tokenizer + positional scheme,
# so re-running the same settings always continues the same model.
PRESET = 'nova'      # or 'xxl', 'titan', ... (nova = BPE 4096 + RoPE flagship)
OUT = '/kaggle/working/aira_model.pt'
# Tip: to continue across Kaggle 12h sessions, upload the previous session's
# aira_model.pt as a Dataset, attach it, and copy it to OUT before training
# (full workflow in docs/KAGGLE_TRAINING.md section 5).

# Optional overrides (defaults come from the preset):
# TOKENIZER = 'bpe'; VOCAB = 4096; ROPE = True
kw = {}
try:
    kw = dict(tokenizer=TOKENIZER, vocab_size=VOCAB, rope=ROPE)
except NameError:
    pass

from train import train
train(PRESET, out_path=OUT, **kw)


In [ ]:
print('Done. Download from the Output tab:', OUT)
print('To use locally: place it next to aira.py as tiny_model.pt')